<a href="https://colab.research.google.com/github/pauljoder/memoire_modigliani_authenticite/blob/main/Memoire_Modigliani_authentification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Authentification stylistique d'œuvres de Modigliani par apprentissage profond

**Projet de mémoire : Master 1 SDHC (Sciences des données, histoire et culture)**

Ce notebook met en œuvre un classifieur d'images visant à distinguer le style de Modigliani de celui d'artistes proches. Il ne s'agit pas de trancher l'authenticité d'une œuvre mais de tester une question circonscrite : **un modèle de vision par ordinateur peut-il reconnaître le style de Modigliani, et que révèle-t-il sur la distinction entre reconnaître un style et authentifier une œuvre ?**

La démarche est volontairement documentée étape par étape, afin d'expliciter mes choix méthodologiques et leurs limites.

## 1. Environnement et bibliothèques

On importe les bibliothèques nécessaires (PyTorch pour l'apprentissage profond, torchvision pour les modèles pré-entraînés et les transformations d'images, ainsi que les outils habituels de manipulation de données et de visualisation). On vérifie ensuite qu'un processeur graphique (GPU) est disponible : ce dernier va accélèrer considérablement l'entraînement.

In [ ]:
# Imports : tous les outils utilisés dans le notebook
import os
import random
import shutil
import copy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Vérification de l'environnement de calcul
print("PyTorch version :", torch.__version__)
print("GPU disponible :", torch.cuda.is_available())
print("Nom du GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun")
print("\nImports chargés.")

## 2. Récupération du corpus

Les images proviennent du jeu de données *Best Artworks of All Time* (Kaggle), qui rassemble des reproductions d'œuvres de cinquante artistes, étiquetées par auteur. L'accès se fait via l'API officielle de Kaggle, ce qui rend la récupération des données traçable et reproductible.

**Note sur les données :** ces images sont des reproductions numériques de qualité variable, et leurs étiquettes sont héritées de WikiArt, elles ne constituent donc pas une vérité-terrain validée scientifiquement.


In [ ]:
from google.colab import files
files.upload()

In [ ]:
# Installer l'outil Kaggle
!pip install -q kaggle

# Placer la clé kaggle.json à l'endroit attendu et la sécuriser
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Clé Kaggle installée.")

In [ ]:
# Téléchargement du dataset "Best Artworks of All Time" depuis Kaggle
!kaggle datasets download -d ikarus777/best-artworks-of-all-time

# Décompression dans un dossier "data"
!unzip -q best-artworks-of-all-time.zip -d data

print("Dataset téléchargé et décompressé.")

## 3. Exploration du corpus

Avant toute modélisation, on inspecte les données. On regarde comment le jeu de données est organisé, combien d'artistes il contient, et surtout combien d'œuvres sont disponibles pour Modigliani -> un point décisif, car la taille du corpus conditionne les choix techniques ultérieurs.

In [ ]:
# Exploration de la structure des dossiers : on repère où sont rangées les images.
# (Le jeu de données imbrique les images dans data/images/images/ et data/resized/resized/.)
print("=== Fichiers à la racine de data ===")
print(os.listdir("data"))

print("\n=== Contenu de data/images ===")
print(os.listdir("data/images"))

print("\n=== Contenu de data/resized ===")
print(os.listdir("data/resized")[:5], "...")  # juste les 5 premiers

In [ ]:
# Lecture des métadonnées : liste des artistes et nombre d'œuvres par artiste.
# Objectif : vérifier le volume disponible pour Modigliani et repérer les artistes proches.
import pandas as pd

artists = pd.read_csv("data/artists.csv")
print("Colonnes disponibles :", list(artists.columns))
print("Nombre d'artistes :", len(artists))
print()
print(artists[["name", "paintings"]].to_string())

In [ ]:
# Vérification des noms exacts des dossiers d'artistes (orthographe, underscores)
# avant de les réutiliser dans le code : évite les erreurs de chemin.
dossier_images = "data/images/images"
noms_dossiers = sorted(os.listdir(dossier_images))
print("Nombre de dossiers d'artistes :", len(noms_dossiers))
print()
for nom in noms_dossiers:
    print(nom)

## 4. Construction des deux classes

On structure les données pour la classification : un dossier `modigliani` et un dossier `non_modigliani`. La classe « non-Modigliani » est composée d'artistes **stylistiquement proches** de l'École de Paris (Picasso, Matisse, Chagall), choisis délibérément pour que la tâche teste la reconnaissance d'un style et non d'une simple différence d'iconographie. Les faux et reproductions ne sont pas utilisés ici : ils serviront uniquement de banc d'essai critique, après l'entraînement.

In [ ]:
import shutil

# Nos choix d'artistes (noms exacts des dossiers)
modigliani = ["Amedeo_Modigliani"]
autres = ["Pablo_Picasso", "Henri_Matisse", "Marc_Chagall"]

source = "data/images/images"
destination = "dataset"  # notre jeu de données structuré

# Créer une fonction qui copie les images d'une liste d'artistes vers une classe
def construire_classe(liste_artistes, nom_classe):
    dossier_classe = os.path.join(destination, nom_classe)
    os.makedirs(dossier_classe, exist_ok=True)
    total = 0
    for artiste in liste_artistes:
        chemin_artiste = os.path.join(source, artiste)
        for image in os.listdir(chemin_artiste):
            shutil.copy(os.path.join(chemin_artiste, image),
                        os.path.join(dossier_classe, f"{artiste}_{image}"))
            total += 1
    print(f"Classe '{nom_classe}' : {total} images")

# Construire les deux classes
construire_classe(modigliani, "modigliani")
construire_classe(autres, "non_modigliani")

## 5. Équilibrage des classes

À ce stade, les deux classes sont déséquilibrées : 193 Modigliani contre près de 900 non-Modigliani. Un tel déséquilibre est un piège : mon modèle pourrait obtenir un bon score en répondant « non-Modigliani » par défaut, sans rien apprendre du style. Pour l'éviter, je ramène la classe majoritaire à 193 images, tirées au hasard. Le tirage est rendu **reproductible** par une graine aléatoire fixée.

In [ ]:
import random

# Graine aléatoire fixée : garantit que le tirage est toujours le même (reproductibilité)
random.seed(42)

dossier_non_modi = "dataset/non_modigliani"
images_non_modi = os.listdir(dossier_non_modi)

print("Avant équilibrage :", len(images_non_modi), "images non-Modigliani")

# On veut en garder autant que de Modigliani (193)
nb_a_garder = len(os.listdir("dataset/modigliani"))

# Tirer au hasard les images à SUPPRIMER (celles en trop)
images_a_supprimer = random.sample(images_non_modi, len(images_non_modi) - nb_a_garder)

for image in images_a_supprimer:
    os.remove(os.path.join(dossier_non_modi, image))

# Vérification finale
print("Après équilibrage :")
print("  modigliani     :", len(os.listdir("dataset/modigliani")), "images")
print("  non_modigliani :", len(os.listdir("dataset/non_modigliani")), "images")

## 6. Préparation des images et découpage train / validation / test

Les images sont redimensionnées en 224×224 pixels (format attendu par ResNet18) et normalisées selon les statistiques d'ImageNet. Sur l'ensemble d'entraînement uniquement, on ajoute de l'**augmentation de données** (retournement et légère rotation aléatoires) : cela va montrer artificiellement plus de variété au modèle et limiter le surapprentissage. Les ensembles de validation et de test, eux, restent intacts, pour une évaluation honnête.

Le corpus est ensuite découpé en **70 % entraînement / 15 % validation / 15 % test**. Ce ratio est adapté à la petite taille du corpus -> il préserve assez d'images pour apprendre tout en gardant des ensembles d'évaluation exploitables.

In [ ]:
# 1. Définir les transformations
# Pour l'ENTRAÎNEMENT : redimensionnement + augmentation (variété artificielle)
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),               # taille attendue par ResNet18
    transforms.RandomHorizontalFlip(),           # retournement gauche-droite aléatoire
    transforms.RandomRotation(15),               # légère rotation aléatoire
    transforms.ToTensor(),                       # conversion en tenseur (format PyTorch)
    transforms.Normalize([0.485, 0.456, 0.406],  # normalisation (valeurs standard ImageNet)
                         [0.229, 0.224, 0.225]),
])

# Pour VALIDATION et TEST : redimensionnement seul, PAS d'augmentation
transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# 2. Charger toutes les images depuis nos dossiers de classes
dataset_complet = datasets.ImageFolder("dataset")
print("Classes détectées :", dataset_complet.classes)
print("Nombre total d'images :", len(dataset_complet))

# 3. Découper en 70 % / 15 % / 15 %
n_total = len(dataset_complet)
n_train = int(0.70 * n_total)
n_val   = int(0.15 * n_total)
n_test  = n_total - n_train - n_val

torch.manual_seed(42)  # reproductibilité du découpage
train_set, val_set, test_set = random_split(dataset_complet, [n_train, n_val, n_test])

print(f"\nDécoupage : {n_train} train / {n_val} validation / {n_test} test")

In [ ]:
# Appliquer les BONNES transformations à chaque sous-ensemble
# (train.dataset est une référence ; on assigne via une petite astuce de copie)


train_set.dataset = copy.deepcopy(dataset_complet)
train_set.dataset.transform = transform_train

val_set.dataset = copy.deepcopy(dataset_complet)
val_set.dataset.transform = transform_eval

test_set.dataset = copy.deepcopy(dataset_complet)
test_set.dataset.transform = transform_eval

# Créer les DataLoaders : ils distribuent les images par paquets (batches)
train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=16, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=16, shuffle=False)

print("DataLoaders créés.")
print("Taille des batches : 16 images")
print("Nombre de batches d'entraînement :", len(train_loader))

## 7. Le modèle : ResNet18 en apprentissage par transfert

On n'entraîne pas un réseau depuis zéro, cela exigerait des millions d'images. On part d'un **ResNet18 pré-entraîné sur ImageNet**, qui sait déjà extraire des formes, des textures, des contours. On **gèle** ces couches et on ne remplace que la dernière couche de décision, réentraînée pour nos deux classes. Ce choix est délibéré : sur un corpus restreint, une architecture plus lourde (ResNet50, Vision Transformers) surapprendrait. ResNet18 est donc le bon outil pour ce volume de données.

In [ ]:
# Choisir l'appareil de calcul : GPU si disponible, sinon CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Calcul sur :", device)

# 1. Charger ResNet18 PRÉ-ENTRAÎNÉ sur ImageNet
modele = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# 2. GELER toutes les couches (on garde leur savoir visuel, on ne le modifie pas)
for param in modele.parameters():
    param.requires_grad = False

# 3. REMPLACER la dernière couche par une nouvelle, adaptée à NOS 2 classes
nb_caracteristiques = modele.fc.in_features
modele.fc = nn.Linear(nb_caracteristiques, 2)  # 2 sorties : modigliani / non_modigliani

# 4. Envoyer le modèle sur le GPU
modele = modele.to(device)

print("Modèle ResNet18 prêt (transfer learning).")
print("Seule la dernière couche sera entraînée.")

## 8. Entraînement

L'apprentissage repose sur trois éléments : une **fonction de perte** (l'entropie croisée, qui mesure l'écart entre prédiction et réalité), un **optimiseur** (Adam, qui ajuste les paramètres pour réduire cette perte, uniquement sur la dernière couche, les autres étant gelées), et un nombre d'**epochs** (passages complets sur les données). À chaque epoch, on mesure la performance sur la validation, pour vérifier que le modèle apprend réellement et ne mémorise pas.

In [ ]:
# Fonction de perte : mesure l'erreur (standard pour la classification)
criterion = nn.CrossEntropyLoss()

# Optimiseur : ajuste SEULEMENT les paramètres de la dernière couche (les seuls "dégelés")
optimizer = optim.Adam(modele.fc.parameters(), lr=0.001)

def entrainer(modele, train_loader, val_loader, nb_epochs):
    historique = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(nb_epochs):
        # --- Phase d'entraînement ---
        modele.train()
        perte_train = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()              # remettre les gradients à zéro
            sorties = modele(images)           # prédiction
            perte = criterion(sorties, labels) # calcul de l'erreur
            perte.backward()                   # rétropropagation
            optimizer.step()                   # ajustement des poids
            perte_train += perte.item()
        perte_train /= len(train_loader)

        # --- Phase de validation (sans apprentissage) ---
        modele.eval()
        perte_val, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                sorties = modele(images)
                perte_val += criterion(sorties, labels).item()
                _, predictions = torch.max(sorties, 1)
                correct += (predictions == labels).sum().item()
                total += labels.size(0)
        perte_val /= len(val_loader)
        precision_val = 100 * correct / total

        historique["train_loss"].append(perte_train)
        historique["val_loss"].append(perte_val)
        historique["val_acc"].append(precision_val)

        print(f"Epoch {epoch+1}/{nb_epochs} | "
              f"perte train: {perte_train:.3f} | "
              f"perte val: {perte_val:.3f} | "
              f"précision val: {precision_val:.1f}%")

    return historique

print("Fonction d'entraînement définie.")

In [ ]:
# Lancement de l'entraînement.
# 10 epochs : hypothèse de départ, confirmée a posteriori par les courbes
# (la performance se stabilise dès la 3e epoch — voir ci-dessous).
historique = entrainer(modele, train_loader, val_loader, nb_epochs=10)

## 9. Lecture des courbes d'apprentissage

On visualise l'évolution de l'apprentissage. À gauche, les deux courbes de perte (entraînement et validation) doivent diminuer ensemble. À droite, la précision de validation doit progresser. Si la perte de validation remontait au-dessus de celle d'entraînement, ce serait le signe d'un surapprentissage, ce qu'on surveille ici.

In [ ]:
epochs = range(1, len(historique["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Courbe 1 : les pertes
ax1.plot(epochs, historique["train_loss"], "o-", label="Perte entraînement")
ax1.plot(epochs, historique["val_loss"], "s-", label="Perte validation")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Perte")
ax1.set_title("Évolution de la perte")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Courbe 2 : la précision de validation
ax2.plot(epochs, historique["val_acc"], "o-", color="green", label="Précision validation")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Précision (%)")
ax2.set_title("Précision sur la validation")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Évaluation honnête sur le jeu de test

On mesure enfin la performance sur les images de **test**, que le modèle n'a jamais vues, ni pour apprendre, ni pour valider.

Au-delà de la précision globale, la **matrice de confusion** révèle quel type d'erreur le modèle commet : confond-il plutôt les Modigliani avec les autres, ou l'inverse ? Cette lecture qualitative est plus instructive que le seul pourcentage.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

modele.eval()
toutes_predictions = []
tous_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        sorties = modele(images)
        _, predictions = torch.max(sorties, 1)
        toutes_predictions.extend(predictions.cpu().numpy())
        tous_labels.extend(labels.numpy())

# Précision globale sur le test
precision_test = 100 * np.mean(np.array(toutes_predictions) == np.array(tous_labels))
print(f"Précision sur le jeu de TEST (jamais vu) : {precision_test:.1f}%\n")

# Rapport détaillé par classe
noms_classes = dataset_complet.classes
print(classification_report(tous_labels, toutes_predictions,
                            target_names=noms_classes, digits=3))

# Matrice de confusion
cm = confusion_matrix(tous_labels, toutes_predictions)
print("Matrice de confusion :")
print(cm)

## 11. Visualiser le regard de la machine : les cartes de chaleur (Grad-CAM)

Jusqu'ici, notre modèle est une **boîte noire** : il décide, mais on ignore pourquoi. La technique Grad-CAM produit une carte de chaleur qui révèle les zones de l'image ayant pesé dans la décision (en chaud, ce que le modèle a regardé ; en froid, ce qu'il a ignoré).

L'enjeu dépasse ici la technique. Si, pour reconnaître un Modigliani, le modèle se concentre sur le **visage allongé** et le **cou étiré**, alors il automatise en partie le geste indiciaire de l'expert. C'est le point de rencontre entre l'histoire de l'art et notre outil computationnel.

Détail : on doit réactiver les gradients sur la dernière couche convolutive (gelée pendant l'entraînement), car Grad-CAM en a besoin pour fonctionner.

In [ ]:
!pip install -q grad-cam
print("Bibliothèque Grad-CAM installée.")

In [ ]:
# Réactiver les gradients sur la couche observée par Grad-CAM
# (on les avait gelés pour l'entraînement, mais Grad-CAM en a besoin pour "voir")
for param in modele.layer4.parameters():
    param.requires_grad = True

print("Gradients réactivés sur layer4 pour la visualisation.")

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# 1. Indiquer au Grad-CAM quelle couche observer (la dernière couche convolutive)
couche_cible = [modele.layer4[-1]]
cam = GradCAM(model=modele, target_layers=couche_cible)

# 2. Fonction pour "dé-normaliser" une image (pour l'afficher avec ses vraies couleurs)
def denormaliser(tenseur):
    moyenne = np.array([0.485, 0.456, 0.406])
    ecart = np.array([0.229, 0.224, 0.225])
    img = tenseur.cpu().numpy().transpose(1, 2, 0)
    img = ecart * img + moyenne
    return np.clip(img, 0, 1)

# 3. Première application : afficher la carte de chaleur sur 4 images du jeu de test.
#    Objectif : vérifier visuellement sur quoi le modèle s'appuie pour décider
#    (idéalement, le visage et le cou allongé pour les Modigliani).
modele.eval()
images_test, labels_test = next(iter(test_loader))  # un premier paquet d'images

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    image = images_test[i].unsqueeze(0).to(device)

    # Prédiction du modèle
    sortie = modele(image)
    _, prediction = torch.max(sortie, 1)
    classe_predite = dataset_complet.classes[prediction.item()]
    classe_reelle = dataset_complet.classes[labels_test[i].item()]

    # Calcul de la carte de chaleur
    carte = cam(input_tensor=image)[0]
    image_visible = denormaliser(images_test[i])
    superposition = show_cam_on_image(image_visible, carte, use_rgb=True)

    # Affichage : image originale en haut, carte de chaleur en bas
    axes[0, i].imshow(image_visible)
    axes[0, i].set_title(f"Réel : {classe_reelle}")
    axes[0, i].axis("off")

    axes[1, i].imshow(superposition)
    axes[1, i].set_title(f"Vu comme : {classe_predite}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

## 12. Outil d'analyse d'une œuvre : verdict, regard et confiance

On définit ici la fonction qui sera utilisée pour tester n'importe quelle image. Elle renvoie trois choses côte à côte : l'**image** analysée, la **carte de chaleur** (où le modèle a regardé), et une **jauge de confiance** (la répartition de la décision entre les deux classes). La jauge matérialise le degré de certitude du modèle : un score proche de 50 % signale une hésitation, un score élevé une forte conviction -> ce qui ne garantit pas que la décision soit juste, comme le montreront les cas critiques.

In [ ]:
def analyser_image_avec_jauge(chemin_image):
    # 1. Charger et préparer l'image
    image_pil = Image.open(chemin_image).convert("RGB")
    image_tensor = transform_eval(image_pil).unsqueeze(0).to(device)

    # 2. Prédiction + probabilités
    modele.eval()
    with torch.no_grad():
        sortie = modele(image_tensor)
        probabilites = torch.softmax(sortie, dim=1)[0]
        _, prediction = torch.max(sortie, 1)

    classe_predite = dataset_complet.classes[prediction.item()]
    p_modigliani = probabilites[0].item() * 100
    p_non = probabilites[1].item() * 100

    # 3. Carte de chaleur
    carte = cam(input_tensor=image_tensor)[0]
    image_visible = denormaliser(image_tensor[0])
    superposition = show_cam_on_image(image_visible, carte, use_rgb=True)

    # 4. Affichage : image | heatmap | jauge
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))

    ax1.imshow(image_visible)
    ax1.set_title("Image déposée")
    ax1.axis("off")

    ax2.imshow(superposition)
    ax2.set_title("Elements regardés")
    ax2.axis("off")

    classes = ["Modigliani", "Non-Modigliani"]
    valeurs = [p_modigliani, p_non]
    couleurs = ["#2E8B57" if p_modigliani >= p_non else "#B0B0B0",
                "#C0504D" if p_non > p_modigliani else "#B0B0B0"]
    ax3.barh(classes, valeurs, color=couleurs)
    ax3.set_xlim(0, 100)
    ax3.set_xlabel("Confiance (%)")
    ax3.set_title(f"Verdict : {classe_predite}")
    for i, v in enumerate(valeurs):
        ax3.text(v + 2, i, f"{v:.1f}%", va="center", fontweight="bold")

    plt.tight_layout()
    plt.show()

    print(f"VERDICT : {classe_predite}")
    print(f"Modigliani {p_modigliani:.1f}% / Non-Modigliani {p_non:.1f}%")

print("Fonction d'analyse avec jauge prête.")

## 13. Tests critiques : confronter le modèle à des cas limites

Notre étape finale du projet. On dépose des œuvres **externes au corpus d'entraînement**, de vrais Modigliani, des artistes proches (Soutine), une contrefacon, et on observe la réaction du modèle.

L'hypothèse : le modèle reconnaît un *style*, pas une *authenticité*. Un faux, conçu pour imiter le style, ou un artiste stylistiquement voisin devraient donc le tromper. Ainsi, cette demonstration nous permet de dire que **reconnaître un style n'est pas authentifier une œuvre.**

Les résultats détaillés sont rassemblés dans le tableau récapitulatif (en annexe).

In [ ]:
from google.colab import files

print("Choisis une image à analyser (un tableau, un faux, une reproduction...)")
fichier = files.upload()

# On dépose une image depuis l'ordinateur, puis on l'analyse
# (verdict + carte de chaleur + jauge de confiance).
nom_fichier = list(fichier.keys())[0]
analyser_image_avec_jauge(nom_fichier)